# Day 047 — Exercise 3: correlation_with_pvalue

**What you'll build:** `correlation_with_pvalue(x, y, method='pearson') -> dict` — compute the Pearson or Spearman correlation coefficient together with its p-value, so you know both the strength and the statistical reliability of the relationship.

**Why it matters:** `df.corr()` gives you r but not a p-value, so you can't tell if a correlation of 0.3 is real or just noise. Adding the p-value answers 'is this relationship real?' — not just 'how strong is it?'.

## Provided: Setup + describe_distribution + test_normality

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def make_sample_data(n: int = 100, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible multi-column dataset for statistics exercises."""
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'normal_col': rng.standard_normal(n).round(3),
        'skewed_col': rng.exponential(2, n).round(3),
        'score_a':    (50 + rng.standard_normal(n) * 10).round(1),
        'score_b':    (70 + rng.standard_normal(n) * 10).round(1),
    })


def describe_distribution(series: pd.Series) -> dict:
    s   = series.dropna()
    q25 = float(s.quantile(0.25))
    q75 = float(s.quantile(0.75))
    return {
        'count':    int(len(s)),
        'mean':     round(float(s.mean()), 4),
        'median':   round(float(s.median()), 4),
        'std':      round(float(s.std(ddof=1)), 4),
        'sem':      round(float(s.sem()), 4),
        'min':      round(float(s.min()), 4),
        'max':      round(float(s.max()), 4),
        'q25':      round(q25, 4),
        'q75':      round(q75, 4),
        'iqr':      round(q75 - q25, 4),
        'skewness': round(float(s.skew()), 4),
        'kurtosis': round(float(s.kurt()), 4),
    }


def test_normality(series: pd.Series, alpha: float = 0.05) -> dict:
    s      = series.dropna()
    stat, p = stats.shapiro(s)
    return {
        'n':          len(s),
        'statistic':  round(float(stat), 4),
        'p_value':    round(float(p), 6),
        'is_normal':  bool(p > alpha),
        'alpha':      alpha,
    }

## Your Implementation

In [ ]:
def correlation_with_pvalue(x: pd.Series, y: pd.Series,
                             method: str = 'pearson') -> dict:
    """
    Pearson or Spearman correlation with p-value.

    Args:
        x, y:   numeric pd.Series (NaN pairs are dropped together)
        method: 'pearson' (linear) or 'spearman' (rank-based)
    Returns:
        dict with keys: method, n, r, p_value, is_significant
        is_significant is True when p_value < 0.05.
    """
    mask  = x.notna() & y.notna()
    x_c, y_c = x[mask], y[mask]
    # TODO: if method == 'pearson':
    #     r, p = stats.pearsonr(x_c, y_c)
    # elif method == 'spearman':
    #     r, p = stats.spearmanr(x_c, y_c)
    # else:
    #     raise ValueError(f"method must be 'pearson' or 'spearman', got {method!r}")
    # TODO: return {
    #     'method':         method,
    #     'n':              len(x_c),
    #     'r':              round(float(r), 4),
    #     'p_value':        round(float(p), 6),
    #     'is_significant': bool(p < 0.05),
    # }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    x = pd.Series([1.0, 2.0, 3.0, 4.0, 5.0])
    y = x * 2 + 3   # perfectly correlated

    # Check 1: defined, returns dict
    try:
        assert 'correlation_with_pvalue' in globals()
        result = correlation_with_pvalue(x, y)
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: correlation_with_pvalue returns dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: all required keys
    try:
        for k in ('method', 'n', 'r', 'p_value', 'is_significant'):
            assert k in result, f'missing key: {k!r}'
        passed += 1; print('\u2705 Check 2: all required keys present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: r is in [-1, 1]
    try:
        r = result['r']
        assert -1.0 <= r <= 1.0, f'r must be in [-1, 1], got {r}'
        passed += 1; print(f'\u2705 Check 3: r={r:.4f} is in [-1, 1]')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: perfect positive correlation → r≈1.0 and is_significant=True
    try:
        assert abs(result['r'] - 1.0) < 1e-4, \
            f'y=2x+3 should give r≈1.0, got {result["r"]}'
        assert result['is_significant'] is True, \
            f'perfect correlation should be significant'
        passed += 1; print(f'\u2705 Check 4: perfect correlation r=1.0, is_significant=True')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: Spearman on same data also gives r≈1.0
    try:
        r_sp = correlation_with_pvalue(x, y, method='spearman')
        assert abs(r_sp['r'] - 1.0) < 1e-4, \
            f'Spearman of y=2x+3 should also be ≈1.0, got {r_sp["r"]}'
        assert r_sp['method'] == 'spearman'
        passed += 1; print(f'\u2705 Check 5: Spearman r={r_sp["r"]:.4f}')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def correlation_with_pvalue(x: pd.Series, y: pd.Series,
                             method: str = 'pearson') -> dict:
    mask  = x.notna() & y.notna()
    x_c, y_c = x[mask], y[mask]
    if method == 'pearson':
        r, p = stats.pearsonr(x_c, y_c)
    elif method == 'spearman':
        r, p = stats.spearmanr(x_c, y_c)
    else:
        raise ValueError(f"method must be 'pearson' or 'spearman', got {method!r}")
    return {
        'method':         method,
        'n':              len(x_c),
        'r':              round(float(r), 4),
        'p_value':        round(float(p), 6),
        'is_significant': bool(p < 0.05),
    }
```

</details>